# Quarto Phase 1F Conv2 Probe Analysis

This notebook consolidates the new Phase 1F probe outputs, compares them against the earlier fc1 baseline and the random conv2 control, and re-ranks the current SAE surfaces in light of the new evidence.

The goals are:
- confirm whether H8 is supported by the new probe results,
- compare the conv2 probe ceiling against the current best SAEs,
- identify which configurations move up or down once threat recovery matters more than global coverage,
- detect missing evaluation coverage and open gaps,
- propose the next experiment matrix in execution order.

In [1]:
from pathlib import Path
import json
import re
from collections import Counter
from html import escape

import plotly.graph_objects as go
from IPython.display import HTML, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "quarto"
EVAL_REGISTRY_PATH = ROOT / "saes" / "quarto" / "eval_registry.json"

CATEGORY_ORDER = [
    "cell_occupancy",
    "cell_attribute",
    "threat_line",
    "threat_square_2x2",
    "offered_piece",
    "global",
    "game_phase",
]
THREAT_CATEGORIES = ("threat_line", "threat_square_2x2")


def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def fmt(value, digits=3):
    if value is None:
        return "-"
    if isinstance(value, float):
        return f"{value:.{digits}f}"
    return str(value)


def display_table(rows, columns=None, title=None, max_rows=None):
    if max_rows is not None:
        rows = rows[:max_rows]
    if not rows:
        display(Markdown(f"**{title or 'Table'}**\n\n_No rows._"))
        return
    if columns is None:
        columns = list(rows[0].keys())

    html = []
    if title:
        html.append(f"<h4>{escape(title)}</h4>")
    html.append("<table>")
    html.append(
        "<thead><tr>"
        + "".join(f"<th>{escape(str(col))}</th>" for col in columns)
        + "</tr></thead>"
    )
    html.append("<tbody>")
    for row in rows:
        html.append(
            "<tr>"
            + "".join(f"<td>{escape(str(row.get(col, '')))}</td>" for col in columns)
            + "</tr>"
        )
    html.append("</tbody></table>")
    display(HTML("".join(html)))


def phase1f_score(run):
    threat_mean = run["threat_mean"]
    return 0.60 * threat_mean + 0.25 * run["coverage"] + 0.15 * run["cell_attribute"]

## 1. Load 1F Results and Prior Benchmarks

This section loads the new Phase 1F probe outputs together with the current Quarto SAE evaluation registry. The target is to put the new conv2 probe ceiling on the same page as the best fc1 and conv2 SAE runs.

In [2]:
fc1_probe = load_json(DATA_DIR / "linear_probe_fc1_amalgam_activations_results.json")
conv2_probe = load_json(
    DATA_DIR / "linear_probe_gorilla_164_conv2_512_amalgam_activations_results.json"
)
conv2_random_probe = load_json(
    DATA_DIR
    / "linear_probe_gorilla_164_conv2_512_amalgam_random_activations_results.json"
)
eval_registry = load_json(EVAL_REGISTRY_PATH)

sae_runs = []
for run_id, entry in eval_registry.items():
    if entry.get("game") != "quarto":
        continue
    if entry.get("bsp_set") != "gorilla":
        continue
    metrics = entry.get("metrics", {})
    per_category = metrics.get("per_category", {})
    if not per_category:
        continue
    row = {
        "run_id": run_id,
        "hook": entry.get("hook", "unknown"),
        "architecture": entry.get("architecture", "unknown"),
        "experiment": entry.get("experiment", "unknown"),
        "coverage": metrics.get("coverage", 0.0),
        "threat_line": per_category.get("threat_line", {}).get("mean_f1", 0.0),
        "threat_square_2x2": per_category.get("threat_square_2x2", {}).get(
            "mean_f1", 0.0
        ),
        "cell_attribute": per_category.get("cell_attribute", {}).get("mean_f1", 0.0),
        "cell_occupancy": per_category.get("cell_occupancy", {}).get("mean_f1", 0.0),
        "game_phase": per_category.get("game_phase", {}).get("mean_f1", 0.0),
        "global": per_category.get("global", {}).get("mean_f1", 0.0),
        "offered_piece": per_category.get("offered_piece", {}).get("mean_f1", 0.0),
        "dead_features_pct": metrics.get("dead_features_pct"),
        "l0": metrics.get("l0"),
    }
    row["threat_mean"] = (row["threat_line"] + row["threat_square_2x2"]) / 2.0
    sae_runs.append(row)

best_fc1_sae = max(
    (row for row in sae_runs if row["hook"] == "fc1"), key=lambda row: row["coverage"]
)
best_conv2_sae = max(
    (row for row in sae_runs if row["hook"] == "conv2"), key=lambda row: row["coverage"]
)

surface_rows = [
    {
        "surface": "fc1 linear probe",
        "type": "probe",
        "hook": "fc1",
        "coverage": fc1_probe["overall"]["coverage"],
        "threat_line": fc1_probe["per_category"]["threat_line"]["mean_f1"],
        "threat_square_2x2": fc1_probe["per_category"]["threat_square_2x2"]["mean_f1"],
        "threat_mean": (
            fc1_probe["per_category"]["threat_line"]["mean_f1"]
            + fc1_probe["per_category"]["threat_square_2x2"]["mean_f1"]
        )
        / 2.0,
    },
    {
        "surface": "conv2 linear probe",
        "type": "probe",
        "hook": "conv2",
        "coverage": conv2_probe["overall"]["coverage"],
        "threat_line": conv2_probe["per_category"]["threat_line"]["mean_f1"],
        "threat_square_2x2": conv2_probe["per_category"]["threat_square_2x2"][
            "mean_f1"
        ],
        "threat_mean": (
            conv2_probe["per_category"]["threat_line"]["mean_f1"]
            + conv2_probe["per_category"]["threat_square_2x2"]["mean_f1"]
        )
        / 2.0,
    },
    {
        "surface": "conv2 random probe",
        "type": "probe-random",
        "hook": "conv2",
        "coverage": conv2_random_probe["overall"]["coverage"],
        "threat_line": conv2_random_probe["per_category"]["threat_line"]["mean_f1"],
        "threat_square_2x2": conv2_random_probe["per_category"]["threat_square_2x2"][
            "mean_f1"
        ],
        "threat_mean": (
            conv2_random_probe["per_category"]["threat_line"]["mean_f1"]
            + conv2_random_probe["per_category"]["threat_square_2x2"]["mean_f1"]
        )
        / 2.0,
    },
    {
        "surface": f"best fc1 SAE ({best_fc1_sae['run_id']})",
        "type": "sae",
        "hook": "fc1",
        "coverage": best_fc1_sae["coverage"],
        "threat_line": best_fc1_sae["threat_line"],
        "threat_square_2x2": best_fc1_sae["threat_square_2x2"],
        "threat_mean": best_fc1_sae["threat_mean"],
    },
    {
        "surface": f"best conv2 SAE ({best_conv2_sae['run_id']})",
        "type": "sae",
        "hook": "conv2",
        "coverage": best_conv2_sae["coverage"],
        "threat_line": best_conv2_sae["threat_line"],
        "threat_square_2x2": best_conv2_sae["threat_square_2x2"],
        "threat_mean": best_conv2_sae["threat_mean"],
    },
]

category_rows = []
for category in CATEGORY_ORDER:
    category_rows.append(
        {
            "category": category,
            "fc1_probe": fmt(fc1_probe["per_category"][category]["mean_f1"]),
            "conv2_probe": fmt(conv2_probe["per_category"][category]["mean_f1"]),
            "conv2_random": fmt(
                conv2_random_probe["per_category"][category]["mean_f1"]
            ),
            "conv2_minus_fc1": fmt(
                conv2_probe["per_category"][category]["mean_f1"]
                - fc1_probe["per_category"][category]["mean_f1"]
            ),
            "conv2_minus_random": fmt(
                conv2_probe["per_category"][category]["mean_f1"]
                - conv2_random_probe["per_category"][category]["mean_f1"]
            ),
        }
    )

display_table(category_rows, title="Per-category probe comparison")
display_table(
    [
        {
            "surface": row["surface"],
            "type": row["type"],
            "hook": row["hook"],
            "coverage": fmt(row["coverage"]),
            "threat_mean": fmt(row["threat_mean"]),
            "threat_line": fmt(row["threat_line"]),
            "threat_square_2x2": fmt(row["threat_square_2x2"]),
        }
        for row in surface_rows
    ],
    title="Key probe and SAE surfaces",
)

sample_bsp_rows = []
for fc1_bsp, conv2_bsp in zip(fc1_probe["per_bsp"], conv2_probe["per_bsp"]):
    if fc1_bsp["bsp_id"] != conv2_bsp["bsp_id"]:
        raise ValueError("BSP ordering mismatch between fc1 and conv2 probe files")
    sample_bsp_rows.append(
        {
            "bsp_id": fc1_bsp["bsp_id"],
            "category": fc1_bsp["category"],
            "fc1_f1": fc1_bsp["f1"],
            "conv2_f1": conv2_bsp["f1"],
            "gain": conv2_bsp["f1"] - fc1_bsp["f1"],
        }
    )
most_improved_bsps = sorted(sample_bsp_rows, key=lambda row: row["gain"], reverse=True)
display_table(
    [
        {
            "bsp_id": row["bsp_id"],
            "category": row["category"],
            "fc1_f1": fmt(row["fc1_f1"], 4),
            "conv2_f1": fmt(row["conv2_f1"], 4),
            "gain": fmt(row["gain"], 4),
        }
        for row in most_improved_bsps
    ],
    title="Top 15 BSP improvements from fc1 to conv2",
    max_rows=15,
)

category,fc1_probe,conv2_probe,conv2_random,conv2_minus_fc1,conv2_minus_random
cell_occupancy,0.998,1.000,0.829,0.002,0.171
cell_attribute,0.611,0.971,0.782,0.359,0.189
threat_line,0.022,0.502,0.019,0.480,0.483
threat_square_2x2,0.113,0.680,0.015,0.567,0.664
offered_piece,0.602,0.789,0.718,0.188,0.071
global,0.698,0.694,0.634,-0.004,0.060
game_phase,0.908,0.936,0.685,0.027,0.251


surface,type,hook,coverage,threat_mean,threat_line,threat_square_2x2
fc1 linear probe,probe,fc1,0.402,0.068,0.022,0.113
conv2 linear probe,probe,conv2,0.789,0.591,0.502,0.680
conv2 random probe,probe-random,conv2,0.428,0.017,0.019,0.015
best fc1 SAE (anakin-batchtopk-k16-exp8-fc1),sae,fc1,0.338,0.088,0.074,0.103
best conv2 SAE (hook-sweep-conv2-topk-k32-exp8-conv2),sae,conv2,0.335,0.093,0.085,0.101


bsp_id,category,fc1_f1,conv2_f1,gain
square_2_1_threat_square,threat_square_2x2,0.0376,0.7315,0.6939
square_2_0_threat_black,threat_square_2x2,0.0701,0.7355,0.6654
square_0_1_threat_tall,threat_square_2x2,0.0870,0.7518,0.6648
square_2_0_threat_square,threat_square_2x2,0.1316,0.7892,0.6576
square_1_0_threat_black,threat_square_2x2,0.0514,0.7033,0.6519
square_1_1_threat_tall,threat_square_2x2,0.0892,0.7177,0.6285
square_0_0_threat_tall,threat_square_2x2,0.1276,0.7441,0.6165
square_2_2_threat_square,threat_square_2x2,0.1146,0.7308,0.6162
square_0_0_threat_square,threat_square_2x2,0.0457,0.6555,0.6098
square_1_0_threat_with_hole,threat_square_2x2,0.0454,0.6531,0.6077


## 2. Normalize Metrics and Re-rank Configurations

The original registry ranking is dominated by overall gorilla coverage. After Phase 1F, the unresolved question is specifically **threat recovery at conv2**, so the ranking below adds a threat-focused score:

`phase1f_score = 0.60 * threat_mean + 0.25 * coverage + 0.15 * cell_attribute`

This keeps overall reconstruction in view, but it intentionally gives more weight to the categories whose availability was newly established by the conv2 probe.

In [3]:
sae_runs_by_coverage = sorted(sae_runs, key=lambda row: row["coverage"], reverse=True)
for idx, row in enumerate(sae_runs_by_coverage, start=1):
    row["coverage_rank"] = idx
    row["phase1f_score"] = phase1f_score(row)

sae_runs_by_phase1f = sorted(
    sae_runs, key=lambda row: row["phase1f_score"], reverse=True
)
for idx, row in enumerate(sae_runs_by_phase1f, start=1):
    row["phase1f_rank"] = idx
    row["rank_delta"] = row["coverage_rank"] - row["phase1f_rank"]

reranked_rows = [
    {
        "new_rank": row["phase1f_rank"],
        "old_rank": row["coverage_rank"],
        "delta": row["rank_delta"],
        "run_id": row["run_id"],
        "hook": row["hook"],
        "arch": row["architecture"],
        "coverage": fmt(row["coverage"]),
        "threat_mean": fmt(row["threat_mean"]),
        "cell_attr": fmt(row["cell_attribute"]),
        "phase1f_score": fmt(row["phase1f_score"]),
    }
    for row in sae_runs_by_phase1f
]
display_table(
    reranked_rows, title="SAE runs re-ranked for post-1F priorities", max_rows=12
)

movers_up = sorted(
    (row for row in sae_runs if row["rank_delta"] > 0),
    key=lambda row: row["rank_delta"],
    reverse=True,
)
movers_down = sorted(
    (row for row in sae_runs if row["rank_delta"] < 0),
    key=lambda row: row["rank_delta"],
)

display_table(
    [
        {
            "run_id": row["run_id"],
            "hook": row["hook"],
            "arch": row["architecture"],
            "old_rank": row["coverage_rank"],
            "new_rank": row["phase1f_rank"],
            "delta": row["rank_delta"],
            "threat_mean": fmt(row["threat_mean"]),
        }
        for row in movers_up
    ],
    title="Largest upward movers under the new threat-focused score",
    max_rows=8,
)

display_table(
    [
        {
            "run_id": row["run_id"],
            "hook": row["hook"],
            "arch": row["architecture"],
            "old_rank": row["coverage_rank"],
            "new_rank": row["phase1f_rank"],
            "delta": row["rank_delta"],
            "threat_mean": fmt(row["threat_mean"]),
        }
        for row in movers_down
    ],
    title="Largest downward movers under the new threat-focused score",
    max_rows=8,
)

hook_summary = []
for hook in sorted({row["hook"] for row in sae_runs}):
    subset = [row for row in sae_runs if row["hook"] == hook]
    hook_summary.append(
        {
            "hook": hook,
            "n_runs": len(subset),
            "mean_coverage": fmt(sum(row["coverage"] for row in subset) / len(subset)),
            "mean_threat": fmt(sum(row["threat_mean"] for row in subset) / len(subset)),
            "best_run": max(subset, key=lambda row: row["phase1f_score"])["run_id"],
        }
    )

display_table(hook_summary, title="Hook-level summary after re-ranking")

fig = go.Figure()
fig.add_bar(
    name="fc1 probe",
    x=[row["category"] for row in category_rows],
    y=[fc1_probe["per_category"][row["category"]]["mean_f1"] for row in category_rows],
)
fig.add_bar(
    name="conv2 probe",
    x=[row["category"] for row in category_rows],
    y=[
        conv2_probe["per_category"][row["category"]]["mean_f1"] for row in category_rows
    ],
)
fig.add_bar(
    name="conv2 random",
    x=[row["category"] for row in category_rows],
    y=[
        conv2_random_probe["per_category"][row["category"]]["mean_f1"]
        for row in category_rows
    ],
)
fig.update_layout(
    barmode="group",
    title="Phase 1F category comparison: fc1 vs conv2 vs random conv2",
    xaxis_title="Category",
    yaxis_title="Mean F1",
    legend_title="Surface",
    width=1100,
    height=500,
)
fig.show()

scatter = go.Figure()
for hook in sorted({row["hook"] for row in sae_runs}):
    subset = [row for row in sae_runs if row["hook"] == hook]
    scatter.add_trace(
        go.Scatter(
            x=[row["coverage"] for row in subset],
            y=[row["threat_mean"] for row in subset],
            mode="markers+text",
            name=f"SAE {hook}",
            text=[row["architecture"] for row in subset],
            textposition="top center",
        )
    )
scatter.add_trace(
    go.Scatter(
        x=[
            fc1_probe["overall"]["coverage"],
            conv2_probe["overall"]["coverage"],
            conv2_random_probe["overall"]["coverage"],
        ],
        y=[
            surface_rows[0]["threat_mean"],
            surface_rows[1]["threat_mean"],
            surface_rows[2]["threat_mean"],
        ],
        mode="markers+text",
        name="Probe baselines",
        text=["fc1 probe", "conv2 probe", "conv2 random"],
        textposition="bottom center",
        marker=dict(symbol="diamond", size=12),
    )
)
scatter.update_layout(
    title="Coverage vs threat recovery across SAEs and probe baselines",
    xaxis_title="Overall gorilla coverage",
    yaxis_title="Threat mean F1",
    width=900,
    height=600,
)
scatter.show()

new_rank,old_rank,delta,run_id,hook,arch,coverage,threat_mean,cell_attr,phase1f_score
1,1,0,anakin-batchtopk-k16-exp8-fc1,fc1,batchtopk,0.338,0.088,0.504,0.213
2,2,0,hook-sweep-conv2-topk-k32-exp8-conv2,conv2,topk,0.335,0.093,0.485,0.212
3,3,0,anakin-topk-k64-exp8-conv2,conv2,topk,0.332,0.086,0.501,0.210
4,4,0,anakin-topk-k64-exp4-conv2,conv2,topk,0.326,0.079,0.500,0.204
5,6,1,anakin-topk-k32-exp4-conv2,conv2,topk,0.309,0.096,0.436,0.200
6,12,6,anakin-jumprelu-t64-exp4-conv2,conv2,jumprelu,0.301,0.094,0.448,0.199
7,9,2,anakin-topk-k32-exp16-fc1,fc1,topk,0.303,0.091,0.431,0.195
8,14,6,anakin-topk-k16-exp8-fc1,fc1,topk,0.294,0.095,0.424,0.194
9,11,2,anakin-s43-topk-k32-exp8-fc1,fc1,topk,0.301,0.086,0.436,0.193
10,10,0,anakin-topk-k32-exp8-fc1,fc1,topk,0.301,0.085,0.436,0.192


run_id,hook,arch,old_rank,new_rank,delta,threat_mean
anakin-jumprelu-t64-exp4-conv2,conv2,jumprelu,12,6,6,0.094
anakin-topk-k16-exp8-fc1,fc1,topk,14,8,6,0.095
anakin-jumprelu-t128-exp8-fc1,fc1,jumprelu,22,19,3,0.081
anakin-jumprelu-t64-exp8-fc1,fc1,jumprelu,23,20,3,0.067
anakin-topk-k32-exp4-fc1,fc1,topk,20,17,3,0.077
anakin-topk-k32-exp16-fc1,fc1,topk,9,7,2,0.091
anakin-s44-topk-k32-exp8-fc1,fc1,topk,15,13,2,0.086
anakin-s43-topk-k32-exp8-fc1,fc1,topk,11,9,2,0.086


run_id,hook,arch,old_rank,new_rank,delta,threat_mean
anakin-batchtopk-k32-exp16-fc1,fc1,batchtopk,7,14,-7,0.073
anakin-jumprelu-t32-exp8-fc1,fc1,jumprelu,8,15,-7,0.073
anakin-vanilla-l1_001-exp8-fc1,fc1,vanilla,18,24,-6,0.053
anakin-batchtopk-k32-exp8-fc1,fc1,batchtopk,5,11,-6,0.072
anakin-p-annealing-exp16-fc1,fc1,p-annealing,19,22,-3,0.056
anakin-batchtopk-k64-exp8-fc1,fc1,batchtopk,24,25,-1,0.065
anakin-jumprelu-t64-exp16-fc1,fc1,jumprelu,17,18,-1,0.071


hook,n_runs,mean_coverage,mean_threat,best_run
conv2,7,0.307,0.081,hook-sweep-conv2-topk-k32-exp8-conv2
fc1,23,0.286,0.070,anakin-batchtopk-k16-exp8-fc1


## 5. Identify Missing Coverage and Evaluation Gaps

The current question is no longer whether conv2 contains threat information. The gaps now are coverage gaps in the **evaluation program**: missing hawk evaluations, missing controls, and missing diagnostics for feature reuse or absorption.

In [4]:
probe_result_names = {path.name for path in DATA_DIR.glob("linear_probe_*results.json")}
bsp_sets_in_registry = Counter(
    entry.get("bsp_set", "unknown")
    for entry in eval_registry.values()
    if entry.get("game") == "quarto"
)
registry_metric_keys = set()
for entry in eval_registry.values():
    if entry.get("game") == "quarto":
        registry_metric_keys.update(entry.get("metrics", {}).keys())
seed_pattern = re.compile(r"-s\d+$")

missing_gap_rows = [
    {
        "gap": "hawk_173 SAE evaluation",
        "status": (
            "missing"
            if bsp_sets_in_registry.get("hawk", 0) == 0
            and bsp_sets_in_registry.get("hawk_173", 0) == 0
            else "present"
        ),
        "evidence": f"quarto eval registry BSP sets: {dict(bsp_sets_in_registry)}",
        "why_it_matters": "We do not yet know whether current SAEs recover the easier count-like threat basis on hawk_173.",
    },
    {
        "gap": "hawk_173 conv2 probe",
        "status": (
            "missing"
            if not any(
                "hawk" in name and "conv2" in name for name in probe_result_names
            )
            else "present"
        ),
        "evidence": "Probe result files under data/quarto/\n"
        + "\n".join(sorted(probe_result_names)),
        "why_it_matters": "Without a conv2 hawk probe, the hawk ceiling is only known at fc1.",
    },
    {
        "gap": "random-model SAE controls",
        "status": (
            "missing"
            if not any(
                "random" in entry.get("tag", "").lower()
                or "random" in entry.get("experiment", "").lower()
                for entry in eval_registry.values()
            )
            else "present"
        ),
        "evidence": "No eval registry entry is tagged as a random-model SAE run.",
        "why_it_matters": "We cannot separate learned SAE features from geometry-only SAE features yet.",
    },
    {
        "gap": "conv2 seed stability",
        "status": (
            "missing"
            if not any(
                seed_pattern.search(row["run_id"]) and row["hook"] == "conv2"
                for row in sae_runs
            )
            else "present"
        ),
        "evidence": "No conv2 run id in the eval registry includes a seed suffix like -s43.",
        "why_it_matters": "Feature-level claims on conv2 remain unstable until we check seed sensitivity.",
    },
    {
        "gap": "feature reuse / one-to-one metrics",
        "status": (
            "missing"
            if "best_feature_per_bsp" not in registry_metric_keys
            else "partial"
        ),
        "evidence": f"Registry metric keys: {sorted(registry_metric_keys)}",
        "why_it_matters": "The current registry tracks coverage, but not how many BSPs collapse onto the same feature.",
    },
]

display_table(missing_gap_rows, title="Programmatically detected gaps")

arch_hook_counts = Counter((row["architecture"], row["hook"]) for row in sae_runs)
arch_hook_rows = [
    {
        "architecture": arch,
        "hook": hook,
        "n_eval_runs": count,
    }
    for (arch, hook), count in sorted(arch_hook_counts.items())
]
display_table(
    arch_hook_rows, title="Observed architecture-hook coverage in the eval registry"
)

gap,status,evidence,why_it_matters
hawk_173 SAE evaluation,missing,quarto eval registry BSP sets: {'gorilla': 30},We do not yet know whether current SAEs recover the easier count-like threat basis on hawk_173.
hawk_173 conv2 probe,missing,Probe result files under data/quarto/ linear_probe_fc1_amalgam_activations_results.json linear_probe_fc1_amalgam_random_activations_results.json linear_probe_gorilla_164_conv2_512_amalgam_activations_results.json linear_probe_gorilla_164_conv2_512_amalgam_random_activations_results.json linear_probe_hawk_fc1_amalgam_activations_results.json linear_probe_hawk_fc1_amalgam_random_activations_results.json,"Without a conv2 hawk probe, the hawk ceiling is only known at fc1."
random-model SAE controls,missing,No eval registry entry is tagged as a random-model SAE run.,We cannot separate learned SAE features from geometry-only SAE features yet.
conv2 seed stability,missing,No conv2 run id in the eval registry includes a seed suffix like -s43.,Feature-level claims on conv2 remain unstable until we check seed sensitivity.
feature reuse / one-to-one metrics,missing,"Registry metric keys: ['board_reconstruction', 'coverage', 'coverage_above_50', 'coverage_above_75', 'dead_features_pct', 'fraction_reconstructable', 'fvu', 'l0', 'l0_std', 'max_f1', 'mean_accuracy', 'median_f1', 'median_feat_freq', 'min_f1', 'mse', 'num_bsps', 'num_reconstructable_bsps', 'per_category']","The current registry tracks coverage, but not how many BSPs collapse onto the same feature."


architecture,hook,n_eval_runs
batchtopk,fc1,4
gated,conv2,1
gated,fc1,2
jumprelu,conv2,1
jumprelu,fc1,4
p-annealing,fc1,2
topk,conv2,5
topk,fc1,9
vanilla,fc1,2


## 6. Generate Tables and Plots for Doc Updates
## 7. Define Next-Experiment Candidates
## 8. Build a Prioritized Experiment Matrix

This final section turns the analysis into compact artifacts that can be copied into the docs and into an explicit next-experiment queue. The emphasis is on experiments that close the gap between the conv2 probe ceiling and current SAE recoverability, not on re-running another broad unsupervised architecture sweep.

In [5]:
doc_update_rows = [
    {
        "metric": "gorilla coverage",
        "fc1_probe": fmt(fc1_probe["overall"]["coverage"]),
        "conv2_probe": fmt(conv2_probe["overall"]["coverage"]),
        "conv2_random": fmt(conv2_random_probe["overall"]["coverage"]),
        "best_conv2_sae": fmt(best_conv2_sae["coverage"]),
    },
    {
        "metric": "threat_line",
        "fc1_probe": fmt(fc1_probe["per_category"]["threat_line"]["mean_f1"]),
        "conv2_probe": fmt(conv2_probe["per_category"]["threat_line"]["mean_f1"]),
        "conv2_random": fmt(
            conv2_random_probe["per_category"]["threat_line"]["mean_f1"]
        ),
        "best_conv2_sae": fmt(best_conv2_sae["threat_line"]),
    },
    {
        "metric": "threat_square_2x2",
        "fc1_probe": fmt(fc1_probe["per_category"]["threat_square_2x2"]["mean_f1"]),
        "conv2_probe": fmt(conv2_probe["per_category"]["threat_square_2x2"]["mean_f1"]),
        "conv2_random": fmt(
            conv2_random_probe["per_category"]["threat_square_2x2"]["mean_f1"]
        ),
        "best_conv2_sae": fmt(best_conv2_sae["threat_square_2x2"]),
    },
]
display_table(doc_update_rows, title="Compact doc-update table")

doc_update_markdown = "\n".join(
    [
        "| Metric | fc1 probe | conv2 probe | conv2 random | best conv2 SAE |",
        "|---|:---:|:---:|:---:|:---:|",
        *[
            f"| {row['metric']} | {row['fc1_probe']} | {row['conv2_probe']} | {row['conv2_random']} | {row['best_conv2_sae']} |"
            for row in doc_update_rows
        ],
    ]
)
display(Markdown("### Markdown-ready summary for docs\n\n" + doc_update_markdown))

experiment_candidates = [
    {
        "experiment": "1G. Evaluate best conv2 SAE on hawk_173",
        "family": "evaluation",
        "expected_value": 10,
        "cost": 2,
        "dependency": "1F done",
        "coverage_gap": "No hawk_173 SAE results yet",
        "why_now": "Tests whether current SAEs recover the easier count-like threat basis before adding new training machinery.",
    },
    {
        "experiment": "1Gb. Evaluate best fc1 SAE on hawk_173",
        "family": "evaluation",
        "expected_value": 7,
        "cost": 2,
        "dependency": "1F done",
        "coverage_gap": "Need a bottleneck comparison against conv2 on the same BSP set",
        "why_now": "Quantifies how much of the hawk basis survives the fc1 bottleneck in the current SAE pipeline.",
    },
    {
        "experiment": "1H. Best-feature reuse / absorption analysis on best conv2 SAE",
        "family": "diagnostic",
        "expected_value": 9,
        "cost": 3,
        "dependency": "Best conv2 SAE checkpoint + matching cache",
        "coverage_gap": "No one-to-one or feature-sharing metric exists yet",
        "why_now": "Explains whether many BSPs collapse onto the same feature despite strong conv2 probe accessibility.",
    },
    {
        "experiment": "3A. Guided TopK SAE pilot on conv2",
        "family": "architecture",
        "expected_value": 8,
        "cost": 7,
        "dependency": "1G and 1H should confirm the unsupervised gap first",
        "coverage_gap": "Current unsupervised SAEs underuse the conv2 probe signal",
        "why_now": "Adds supervision only after verifying that the unsupervised failure is real and not just an evaluation blind spot.",
    },
    {
        "experiment": "3B. Feature-anchored conv2 SAE pilot",
        "family": "architecture",
        "expected_value": 7,
        "cost": 7,
        "dependency": "Probe directions and 1H diagnostics",
        "coverage_gap": "No anchor-based identifiability experiment exists yet",
        "why_now": "Targets the representation-alignment problem more softly than Guided SAE.",
    },
    {
        "experiment": "Conv2 seed-stability replication",
        "family": "stability",
        "expected_value": 6,
        "cost": 5,
        "dependency": "Choose the best conv2 checkpoint family first",
        "coverage_gap": "No multi-seed conv2 evidence",
        "why_now": "Needed before making strong feature-level claims on any conv2 architecture.",
    },
    {
        "experiment": "Broad new unsupervised architecture sweep",
        "family": "architecture",
        "expected_value": 3,
        "cost": 9,
        "dependency": "Not recommended yet",
        "coverage_gap": "Low current value after Anakin + 1F",
        "why_now": "Defer: current evidence says the main missing ingredient is not another unsupervised variant of the same family.",
    },
]
for row in experiment_candidates:
    row["priority_score"] = row["expected_value"] / row["cost"]

prioritized_experiments = sorted(
    experiment_candidates,
    key=lambda row: (row["priority_score"], row["expected_value"]),
    reverse=True,
)

display_table(
    [
        {
            "priority": idx,
            "experiment": row["experiment"],
            "family": row["family"],
            "value": row["expected_value"],
            "cost": row["cost"],
            "value/cost": fmt(row["priority_score"]),
            "dependency": row["dependency"],
        }
        for idx, row in enumerate(prioritized_experiments, start=1)
    ],
    title="Prioritized experiment matrix",
)

matrix = go.Figure()
for family in sorted({row["family"] for row in prioritized_experiments}):
    subset = [row for row in prioritized_experiments if row["family"] == family]
    matrix.add_trace(
        go.Scatter(
            x=[row["cost"] for row in subset],
            y=[row["expected_value"] for row in subset],
            mode="markers+text",
            name=family,
            text=[row["experiment"] for row in subset],
            textposition="top center",
        )
    )
matrix.update_layout(
    title="Next-experiment matrix: expected value vs cost",
    xaxis_title="Estimated cost (relative units)",
    yaxis_title="Expected information value (relative units)",
    width=950,
    height=600,
)
matrix.show()

best_next_experiment = prioritized_experiments[0]["experiment"]
display(
    Markdown(
        f"### Notebook recommendation\n\n**Next experiment:** {best_next_experiment}"
    )
)

metric,fc1_probe,conv2_probe,conv2_random,best_conv2_sae
gorilla coverage,0.402,0.789,0.428,0.335
threat_line,0.022,0.502,0.019,0.085
threat_square_2x2,0.113,0.680,0.015,0.101


### Markdown-ready summary for docs

| Metric | fc1 probe | conv2 probe | conv2 random | best conv2 SAE |
|---|:---:|:---:|:---:|:---:|
| gorilla coverage | 0.402 | 0.789 | 0.428 | 0.335 |
| threat_line | 0.022 | 0.502 | 0.019 | 0.085 |
| threat_square_2x2 | 0.113 | 0.680 | 0.015 | 0.101 |

priority,experiment,family,value,cost,value/cost,dependency
1,1G. Evaluate best conv2 SAE on hawk_173,evaluation,10,2,5.000,1F done
2,1Gb. Evaluate best fc1 SAE on hawk_173,evaluation,7,2,3.500,1F done
3,1H. Best-feature reuse / absorption analysis on best conv2 SAE,diagnostic,9,3,3.000,Best conv2 SAE checkpoint + matching cache
4,Conv2 seed-stability replication,stability,6,5,1.200,Choose the best conv2 checkpoint family first
5,3A. Guided TopK SAE pilot on conv2,architecture,8,7,1.143,1G and 1H should confirm the unsupervised gap first
6,3B. Feature-anchored conv2 SAE pilot,architecture,7,7,1.000,Probe directions and 1H diagnostics
7,Broad new unsupervised architecture sweep,architecture,3,9,0.333,Not recommended yet


### Notebook recommendation

**Next experiment:** 1G. Evaluate best conv2 SAE on hawk_173

## Conclusion

H8 is confirmed: threat information is linearly accessible in `conv2` and largely lost by `fc1`.

The next experiment should be **hawk_173 SAE evaluation on the best conv2 checkpoint**, followed immediately by a **feature reuse / absorption analysis** on that same checkpoint. Those two steps close the main uncertainty that remains after 1F: whether current unsupervised SAEs are failing because the task is still too hard, or because multiple BSPs are collapsing onto shared conv2 features.

A broad new sweep of standard unsupervised SAE variants is **not** the best next move. Anakin already showed that architecture choice inside the current family is second-order. What is still missing is targeted evaluation and targeted diagnostics, then, if needed, a guided or anchored conv2 SAE pilot.